# Poster2Movie — 02. Reseña: cartel → sinopsis

Versión **base**: Qwen3-VL-2B-Instruct sin entrenar, con una instrucción (detalles en `p2m/resena.py`).

**Entorno: GPU (T4).**

## Preparar el entorno

Monta tu Google Drive (ahí vive el dataset, `poster2movie_datos/`), clona el repo e instala lo necesario. Si el dataset te lo compartieron, agrega la carpeta `poster2movie_datos` a *Mi unidad* como acceso directo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"
# EQUIPOS: el dataset compartido es solo de lectura; guarden sus modelos y resultados en una
# carpeta suya quitando el # de la siguiente línea (cambien "equipoX" por su equipo):
# os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_equipoX"
SALIDAS = os.environ.get("P2M_SALIDAS", os.environ["P2M_DATOS"])
REPO = "AndreaHortaHernandez/poster2movie"
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo
!pip install -q -r requirements.txt

## 1. Probar con un cartel

In [ ]:
from google.colab import files
from PIL import Image
from p2m.resena import Resenador

resenador = Resenador()
for nombre in files.upload():
    print(resenador.generar(Image.open(nombre).convert('RGB')))

## 2. Evaluar en test (200 carteles, ~15-20 min)

ROUGE-L y BERTScore contra la sinopsis real. Con `--generos predichos` usa los géneros del clasificador en vez de los reales. Resultados en Drive: `poster2movie_datos/resultados/resena/` (incluye los textos generados).

In [ ]:
del resenador  # libera la GPU
!python -m p2m.evaluar_resena --cuantos 200

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
pd.read_csv(SALIDAS + "/resultados/resena/base_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(10)

## 3. Entrenar con LoRA

Afina las capas de atención del modelo de lenguaje (la torre de visión y el resto del modelo quedan congelados) sobre pares cartel → `overview` del split **train**. Por default toma una muestra de `CUANTOS_TRAIN` películas (no hace falta el train completo para notar un cambio); súbele si el tiempo lo permite.

**Nota:** esta sección no se pudo probar con GPU real antes de subirla — la primera corrida en Colab puede necesitar ajustes (nombres de capas, memoria, tamaño de lote). Revisa la celda de diagnóstico antes de armar el `LoraConfig`.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForImageTextToText, AutoProcessor
from p2m.resena import MODELO_RESENA, construir_instruccion

procesador = AutoProcessor.from_pretrained(MODELO_RESENA)
# bfloat16 (no float16, aunque sea mas lento en la T4): mas estable para entrenar,
# evita NaNs en el gradiente sin necesitar un GradScaler.
modelo_base = AutoModelForImageTextToText.from_pretrained(MODELO_RESENA, dtype=torch.bfloat16).to("cuda")

# Diagnostico: nombres reales de las capas lineales de la PRIMERA capa decodificadora.
# Sirve para confirmar los target_modules de abajo y ver si el nombre de la torre de
# vision se puede distinguir del modelo de lenguaje (varia segun version de transformers).
for nombre, modulo in modelo_base.named_modules():
    if isinstance(modulo, torch.nn.Linear) and ".layers.0." in nombre:
        print(nombre)

In [ ]:
config_lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # Si el diagnostico de arriba muestra las mismas capas tambien dentro de la torre de
    # vision ("visual"/"vision" en el nombre), descomenta para no tocarla (necesita
    # peft reciente):
    # exclude_modules=r".*vis.*",
)
modelo = get_peft_model(modelo_base, config_lora)
modelo.print_trainable_parameters()

### Datos de entrenamiento

Una muestra del split **train** (no se toca test ni val). Se eligen ids primero y luego se filtra al recorrer `datos.peliculas(...)`, para no cargar todas las imágenes del train completo a memoria.

In [ ]:
import random
from p2m import datos

CUANTOS_TRAIN = 100  # PRUEBA RAPIDA: confirma que no truena antes de subir a 3000+
SEMILLA = 123

metadata_train = datos.metadata("train")
ids_entrenamiento = set(
    metadata_train.sample(n=min(CUANTOS_TRAIN, len(metadata_train)), random_state=SEMILLA)["id"]
)
ejemplos = [
    f for f in datos.peliculas("train", ["id", "imagen", "overview", "generos"])
    if f["id"] in ids_entrenamiento
]
print(f"{len(ejemplos)} peliculas de entrenamiento")

In [ ]:
def preparar_ejemplo(cartel, generos, overview):
    """Arma un ejemplo de entrenamiento: mismo prompt que en inferencia (Resenador),
    con el overview real pegado como respuesta y el prompt enmascarado (-100) en labels."""
    mensajes = [{"role": "user", "content": [
        {"type": "image", "image": cartel},
        {"type": "text", "text": construir_instruccion(list(generos))},
    ]}]
    entrada = procesador.apply_chat_template(
        mensajes, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    )
    objetivo = procesador.tokenizer(overview.strip(), add_special_tokens=False, return_tensors="pt")
    eos = torch.tensor([[procesador.tokenizer.eos_token_id]])

    lote = dict(entrada)
    lote["input_ids"] = torch.cat([entrada["input_ids"], objetivo["input_ids"], eos], dim=1)
    lote["attention_mask"] = torch.cat(
        [entrada["attention_mask"], torch.ones_like(objetivo["input_ids"]), torch.ones_like(eos)], dim=1
    )
    lote["labels"] = torch.cat(
        [torch.full_like(entrada["input_ids"], -100), objetivo["input_ids"], eos], dim=1
    )
    return {k: v.to("cuda") for k, v in lote.items()}

### Entrenar

Lote efectivo de `ACUMULAR` carteles (gradiente acumulado, no batch real: cada cartel trae un número distinto de tokens de imagen). ~1-2 horas estimadas para 3000 ejemplos en una T4 — **esto es un estimado, nadie lo cronometró todavía**; si se ve muy lento, baja `CUANTOS_TRAIN`.

In [ ]:
from torch.optim import AdamW
import time

EPOCAS = 1
ACUMULAR = 8
TASA_APRENDIZAJE = 2e-4

optimizador = AdamW([p for p in modelo.parameters() if p.requires_grad], lr=TASA_APRENDIZAJE)
modelo.train()

random.shuffle(ejemplos)
inicio, paso = time.time(), 0
for i, f in enumerate(ejemplos, start=1):
    cartel = datos.abrir_cartel(f["imagen"])
    lote = preparar_ejemplo(cartel, f["generos"], f["overview"])
    salida = modelo(**lote)
    (salida.loss / ACUMULAR).backward()
    if i % ACUMULAR == 0:
        optimizador.step()
        optimizador.zero_grad()
        paso += 1
        if paso % 20 == 0:
            print(f"paso {paso} ({i}/{len(ejemplos)}): loss {salida.loss.item():.3f} "
                  f"({time.time()-inicio:.0f}s)", flush=True)

## 4. Guardar y subir a Hugging Face

Se fusiona el adaptador LoRA con el modelo base (`merge_and_unload`) y se sube el modelo completo a Hugging Face Hub — así `evaluar_resena.py --modelo <repo>` lo carga igual que el base, sin tocar la interfaz de `Resenador`. Mismo flujo que con el traductor.

In [ ]:
modelo_final = modelo.merge_and_unload()
RUTA_MODELO = "/content/poster2movie_resena_lora"
modelo_final.save_pretrained(RUTA_MODELO)
procesador.save_pretrained(RUTA_MODELO)

In [ ]:
from huggingface_hub import login
login()  # pega un token de Hugging Face con permiso de escritura

REPO_HF = "tu-usuario-hf/poster2movie-resena-equipo4-prueba"  # cambia "tu-usuario-hf"; quita "-prueba" cuando ya sea la version final
modelo_final.push_to_hub(REPO_HF)
procesador.push_to_hub(REPO_HF)

## 5. Evaluar la versión con LoRA

Libera la GPU del entrenamiento antes de evaluar (el evaluador carga su propia copia del modelo, desde el repo de Hugging Face que acabas de subir).

In [ ]:
del modelo, modelo_final, modelo_base, procesador
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!python -m p2m.evaluar_resena --cuantos 5 --nombre equipo4_lora_prueba --modelo tu-usuario-hf/poster2movie-resena-equipo4-prueba

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 300)
print(pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_prueba_metricas.csv").to_string(index=False))
pd.read_csv(SALIDAS + "/resultados/resena/equipo4_lora_prueba_ejemplos.csv")[["titulo", "generada", "real", "BERTScore"]].head(5)